In [1]:
import pandas as pd
import numpy as np
import glob, os, shutil
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import Normalize as MplNormalize


def getFiles(path, limit=None, shuffle=False):
    target = sorted(glob.glob(os.path.join(path, '*')))
    if shuffle:
        np.random.shuffle(target) 
    return target[:limit]

def formatAxis(img):
    return np.transpose(img, (0, 2, 1))

def setFolder(path):
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path)

def showTile(img=None, mask=None, save=None):
    if img is None and mask is None:
        return print("Erro: Forneça pelo menos 'img' ou 'mask'.")

    ref_vol = img if img is not None else mask
    mid_x = ref_vol.shape[0] // 2
    mid_y = ref_vol.shape[1] // 2
    mid_z = ref_vol.shape[2] // 2

    def get_slices(vol):
        if vol is None:
            return None
        
        s_x = np.array(vol[mid_x, :, :]) # Plano YZ
        s_y = np.array(vol[:, mid_y, :]) # Plano XZ
        s_z = np.array(vol[:, :, mid_z]) # Plano XY
        return [s_x, np.rot90(s_z, -1), s_y]

    img_slices  = get_slices(img)
    mask_slices = get_slices(mask)
    cmap_mask_only    = ListedColormap(['black', 'red', 'green', 'blue'])
    cmap_mask_overlay = ListedColormap([(0, 0, 0, 0), 'red', 'green', 'blue'])

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    titles    = [f'Slice X={mid_x}', f'Slice Y={mid_y}', f'Slice Z={mid_z}']

    for i, ax in enumerate(axes):
        if img is not None:
            ax.imshow(img_slices[i], cmap='gray')
            
        if mask is not None:
            if img is not None:
                ax.imshow(mask_slices[i], cmap=cmap_mask_overlay, vmin=0, vmax=3, alpha=0.6)
            else:
                ax.imshow(mask_slices[i], cmap=cmap_mask_only, vmin=0, vmax=3)
        
        ax.set_title(titles[i])

    plt.tight_layout()

    if save:
        plt.savefig(save, bbox_inches='tight', dpi=300)
        return plt.close(fig)

    plt.show()


def show3DCube(ax, volume, label, x_ratio=0.1, y_ratio=0.9, z_ratio=0.9, stride=1):
    nx, ny, nz = volume.shape
    pos_x, pos_y, pos_z = int(nx * x_ratio), int(ny * y_ratio), int(nz * z_ratio)

    cmap = plt.cm.gray
    norm = MplNormalize(vmin=volume.min(), vmax=volume.max())

    def plot_plane(axis_to_fix, fixed_pos):
        if axis_to_fix == 'y':    # Plano XZ
            ranges_dim1 = [(0, pos_x + 1), (pos_x, nx)]
            ranges_dim2 = [(0, pos_z + 1), (pos_z, nz)]
        elif axis_to_fix == 'x':  # Plano YZ
            ranges_dim1 = [(0, pos_y + 1), (pos_y, ny)]
            ranges_dim2 = [(0, pos_z + 1), (pos_z, nz)]
        else:                     # Plano XY (z)
            ranges_dim1 = [(0, pos_x + 1), (pos_x, nx)]
            ranges_dim2 = [(0, pos_y + 1), (pos_y, ny)]

        for start1, end1 in ranges_dim1:
            for start2, end2 in ranges_dim2:
                arr1, arr2 = np.arange(start1, end1), np.arange(start2, end2)

                if axis_to_fix == 'y':
                    X, Z = np.meshgrid(arr1, arr2, indexing='ij')
                    Y = np.full_like(X, fixed_pos)
                    Z_plot = nz - Z
                    data = volume[start1:end1, fixed_pos, start2:end2]
                elif axis_to_fix == 'x':
                    Y, Z = np.meshgrid(arr1, arr2, indexing='ij')
                    X = np.full_like(Y, fixed_pos)
                    Z_plot = nz - Z
                    data = volume[fixed_pos, start1:end1, start2:end2]
                else:
                    X, Y = np.meshgrid(arr1, arr2, indexing='ij')
                    Z_plot = np.full_like(X, nz - fixed_pos)
                    data = volume[start1:end1, start2:end2, fixed_pos]

                ax.plot_surface(X, Y, Z_plot, facecolors=cmap(norm(data)), shade=False, antialiased=False, linewidth=0, rstride=stride, cstride=stride)

    plot_plane('y', pos_y) # Parede XZ
    plot_plane('x', pos_x) # Parede YZ
    plot_plane('z', pos_z) # Chão XY

    ax.set_xlim(0, nx)
    ax.set_ylim(0, ny)
    ax.set_zlim(0, nz)
    ax.set_box_aspect([1, 1, 1])
    ax.set_axis_off()
    ax.set_title(label, fontsize=14, fontweight='bold', loc='left')
    ax.view_init(elev=20, azim=-45)


def showSteps(steps, save=None):
    fig = plt.figure(figsize=(18, 12))
    for i, (volume, label) in enumerate(steps):
        ax = fig.add_subplot(2, 3, i + 1, projection='3d')
        show3DCube(ax, volume, label)
        
    plt.tight_layout()

    if save:
        plt.savefig(save, bbox_inches='tight', dpi=300)

    plt.show()

In [2]:
import numpy as np
from tqdm import tqdm
import scipy.ndimage as ndimage
import os, json


class SyntheticGenerator:
    def __init__(self, shape=(128, 128, 128)):
        # ── Image Format ─────────────────────────────────────────────
        self.margin = 64                  # Buffer para absorver dobras extremas nas bordas com segurança
        self.finalShape = shape           # (nx, ny, nz) final output volume size

        # ── Refletividade (Estratigrafia) ────────────────────────────
        self.layerRange = (100, 230)      # Qtd de camadas. ↑ Imagem cheia de linhas finas. ↓ Blocos grossos e lisos.
        self.layerThickness = (1, 2)      # Espessura. ↑ Camadas mais grossas. ↓ Camadas bem fininhas.

        # ── Dobramentos (Folding) ────────────────────────────────────
        self.foldCount = (15, 30)         # Qtd de dobras. ↑ Imagem muito ondulada. ↓ Terreno plano.
        self.foldSigma = (8, 44)          # Largura da dobra. ↑ Dobras largas e suaves. ↓ Dobras curtas e apertadas.
        self.foldAmplitude = (-17, 17)    # Altura da dobra. ↑ Picos e vales extremos. ↓ Dobras rasas.
        self.foldDamping   = 1.5          # Perda de força. ↑ A dobra some rápido no fundo. ↓ A dobra desce até a base.
        self.foldBaseShift = (-1.6, 1.6)  # Posição Z. ↑/↓ Sobe ou desce o desenho inteiro na imagem.

        # ── Cisalhamento / Inclinação (Shearing) ─────────────────────
        self.shearOffset   = (-2.8, 2.8)  # Deslocamento lateral. ↑/↓ Empurra todo o bloco para o lado.
        self.shearGradient = (-0.1, 0.1)  # Inclinação (Mergulho). ↑ Camadas ficam na diagonal. ↓ Ficam na horizontal.

        # ── Falhas (Faulting) ────────────────────────────────────────
        self.faultCount = (4, 7)          # Qtd de falhas. ↑ Imagem toda fraturada. ↓ Imagem mais inteira.
        self.faultThrow = (0, 22)         # Tamanho do degrau. ↑ Desencontro gigante nas linhas. ↓ Quebra quase invisível.
        self.faultDipAngle = (20, 75)     # Ângulo. ↑ Falha quase em pé (vertical). ↓ Falha deitada.
        
        self.faultRoughness  = 3.3        # Textura do corte. ↑ Corte tremido/áspero. ↓ Corte liso como navalha.
        self.faultRoughSigma = 4.5        # Tamanho da tremedeira. ↑ Ondas grandes na falha. ↓ Ondinhas curtas.
        self.faultDecaySigma = (33, 83)   # Arrasto. ↑ A linha entorta muito antes de quebrar. ↓ Quebra seca.

        self.faultZoneWidth  = 1.2        # Espessura do rótulo. ↑ A máscara da falha fica grossa. ↓ Fica fina.
        self.faultThreshold  = 0.8        # Filtro de rótulo. ↑ Marca só falha grande. ↓ Marca qualquer rachadurazinha.

        self.faultCurveProb  = 0.30       # Chance de curvar. ↑ Falha faz formato de colher (lístrica). ↓ Falha reta.
        self.faultCurveMax   = 6.7        # Força da curva. ↑ Curva muito fechada. ↓ Curva leve.

        # ── Assinatura Sísmica (Wavelet) ─────────────────────────────
        self.waveletFreq = (81, 117)      # Resolução. ↑ Imagem super nítida. ↓ Imagem borrada e grossa.
        self.waveletDuration = 0.08       # "Eco" do sinal. ↑ O traço borra verticalmente. ↓ Sinal limpo e curto.
        self.waveletDt = 0.002            # Amostragem. ↑ Imagem pode ficar pixelada/serrilhada. ↓ Imagem contínua.

        # ── Ruído Final (Noise) ──────────────────────────────────────
        self.noiseLevel = (0.00, 0.10)    # Chuvisco. ↑ Imagem cheia de ruído (ruim). ↓ Imagem limpa (perfeita).

        self.nx = self.finalShape[0] + 2 * self.margin
        self.ny = self.finalShape[1] + 2 * self.margin
        self.nz = self.finalShape[2] + 2 * self.margin
        self.shape = (self.nx, self.ny, self.nz)

    def get(self):
        data = self.genReflectivity()
        data = self.applyFolding(data)
        data = self.applyShearing(data)
        data, mask = self.applyFaulting(data)
        image = self.applyWavelet(data)
        image = self.applyNoise(image)

        image = self.crop(image)
        mask  = self.crop(mask)
        image = (image - np.mean(image)) / (np.std(image) + 1e-8)
        return image.astype(np.float32), mask.astype(np.uint8)

    def dataset(self, n=200, outputDir="output"):
        imgDir = os.path.join(outputDir, "images")
        mskDir = os.path.join(outputDir, "masks")
        setFolder(imgDir)
        setFolder(mskDir)

        for i in tqdm(range(n), desc="Generating dataset"):
            image, mask = self.get()
            image, mask = formatAxis(image), formatAxis(mask)
            
            np.save(os.path.join(imgDir, f"img_{i:04d}.npy"), image)
            np.save(os.path.join(mskDir, f"img_{i:04d}.npy"), mask)

    def genReflectivity(self):
        """Create 1D layered reflectivity tiled across the volume."""
        r1d = np.zeros(self.nz)
        nLayers = np.random.randint(*self.layerRange)

        for _ in range(nLayers):
            pos = np.random.randint(0, self.nz)
            thickness = np.random.randint(*self.layerThickness)
            r1d[pos : pos + thickness] = np.random.uniform(-1, 1)

        return np.tile(r1d, (self.nx, self.ny, 1))
        
    def applyFolding(self, reflectivity):
        """Deform layers with rotated anisotropic Gaussian folds."""
        x = np.arange(self.nx)
        y = np.arange(self.ny)
        xx, yy = np.meshgrid(x, y, indexing="ij")

        a0 = np.random.uniform(*self.foldBaseShift)
        nGaussians = np.random.randint(*self.foldCount)
        shift2d    = np.zeros((self.nx, self.ny))

        for _ in range(nGaussians):
            x0 = np.random.uniform(-self.nx * 0.3, self.nx * 1.3)
            y0 = np.random.uniform(-self.ny * 0.3, self.ny * 1.3)
            sigmaX = np.random.uniform(*self.foldSigma)
            sigmaY = np.random.uniform(*self.foldSigma)
            theta  = np.random.uniform(0, np.pi)
            amp = np.random.uniform(*self.foldAmplitude)

            dx = xx - x0
            dy = yy - y0
            cosT, sinT = np.cos(theta), np.sin(theta)
            u = cosT * dx + sinT * dy
            v = -sinT * dx + cosT * dy
            shift2d += amp * np.exp(-(u**2 / (2 * sigmaX**2) + v**2 / (2 * sigmaY**2)))

        zGrid = np.arange(self.nz)
        damping = self.foldDamping * zGrid / (self.nz - 1)
        s1 = a0 + shift2d[:, :, np.newaxis] * damping

        ix, iy, iz = np.indices(self.shape)
        return ndimage.map_coordinates(reflectivity, [ix, iy, iz + s1], order=3, mode="nearest")

    def applyShearing(self, reflectivity):
        """Apply linear shear (dip/tilt) along X and Y axes."""
        e0 = np.random.uniform(*self.shearOffset)
        f  = np.random.uniform(*self.shearGradient)
        g  = np.random.uniform(*self.shearGradient)

        ix, iy, iz = np.indices(self.shape)
        s2 = e0 + f * ix + g * iy
        return ndimage.map_coordinates(reflectivity, [ix, iy, iz + s2], order=3, mode="nearest")

    def applyFaulting(self, reflectivity):
        """Inject faults with displacement and produce binary mask."""
        masks = np.zeros(self.shape, dtype=np.uint8)
        model = np.copy(reflectivity)

        numFaults  = np.random.randint(*self.faultCount)
        ix, iy, iz = np.indices(self.shape)

        for i in range(numFaults):
            p0 = np.random.uniform(0.15, 0.85, 3) * np.array(self.shape)

            dip_angle  = np.random.uniform(*self.faultDipAngle)
            dip_rad    = np.deg2rad(dip_angle)
            strike_rad = np.random.uniform(0, 2 * np.pi)
            nx = np.sin(dip_rad) * np.cos(strike_rad)
            ny = np.sin(dip_rad) * np.sin(strike_rad)
            nz = np.cos(dip_rad) * np.random.choice([-1.0, 1.0])
            normal = np.array([nx, ny, nz])

            strike = np.array([-normal[1], normal[0], 0.0])
            strikeNorm = np.linalg.norm(strike)
            strike = np.array([1.0, 0.0, 0.0]) if strikeNorm < 1e-6 else strike / strikeNorm
            dip = np.cross(normal, strike)
            dip /= np.linalg.norm(dip)

            dx = ix - p0[0]
            dy = iy - p0[1]
            dz = iz - p0[2]

            distStrike = strike[0] * dx + strike[1] * dy + strike[2] * dz
            distDip = dip[0] * dx + dip[1] * dy + dip[2] * dz
            bend    = 0.0
            
            if np.random.random() < self.faultCurveProb:
                max_dist = max(self.shape) / 1.5 
                intensidade_base = np.random.uniform(self.faultCurveMax * 0.5, self.faultCurveMax)
                direcao = np.random.choice([-1.0, 1.0])
                curve_intensity = intensidade_base * direcao
                bend = curve_intensity * ((distDip / max_dist) ** 2)

            noisePlane = ndimage.gaussian_filter(np.random.normal(0, 1, self.shape), sigma=self.faultRoughSigma) * self.faultRoughness
            distPlane  = normal[0] * dx + normal[1] * dy + normal[2] * dz + noisePlane - bend
            maxDisp  = np.random.uniform(*self.faultThrow)
            throwMap = self.computeThrowMap(distStrike, distDip, maxDisp)

            shift_x = np.zeros_like(ix, dtype=float)
            shift_y = np.zeros_like(iy, dtype=float)
            shift_z = np.zeros_like(iz, dtype=float)

            hw = distPlane > 0
            shift_x[hw] = throwMap[hw] * dip[0]
            shift_y[hw] = throwMap[hw] * dip[1]
            shift_z[hw] = throwMap[hw] * dip[2]

            ixShifted = ix.astype(float) + shift_x
            iyShifted = iy.astype(float) + shift_y
            izShifted = iz.astype(float) + shift_z

            model = ndimage.map_coordinates(model, [ixShifted, iyShifted, izShifted], order=1, mode="nearest")
            masks = ndimage.map_coordinates(masks, [ixShifted, iyShifted, izShifted], order=0, mode="constant", cval=0)
            faultZone = (np.abs(distPlane) <= self.faultZoneWidth) & (np.abs(throwMap) > self.faultThreshold)
            masks[faultZone] = 1

        return model, masks

    def computeThrowMap(self, distStrike, distDip, maxDisp):
        """Compute displacement map for a single fault (gaussian or linear decay)."""
        if np.random.random() < 0.5:
            sigmaPlane = np.random.uniform(*self.faultDecaySigma)
            return maxDisp * np.exp(-(distStrike**2 + distDip**2) / (2 * sigmaPlane**2))

        planeExtent = np.sqrt(self.nx**2 + self.ny**2 + self.nz**2)
        direction   = np.random.choice([-1, 1])
        return maxDisp * np.clip(0.5 + direction * distDip / planeExtent, 0, 1)

    def applyWavelet(self, model):
        """Convolve with a Ricker wavelet along the Z axis."""
        f = np.random.uniform(*self.waveletFreq)
        t = np.arange(-self.waveletDuration, self.waveletDuration, self.waveletDt)
        wavelet = (1 - 2 * (np.pi * f * t) ** 2) * np.exp(-((np.pi * f * t) ** 2))
        return ndimage.convolve1d(model, wavelet, axis=2)

    def applyNoise(self, image):
        """Add band-limited Gaussian noise scaled to signal amplitude."""
        scale = np.random.uniform(*self.noiseLevel) * np.std(image)
        noise = np.random.normal(0.0, 1.0, image.shape)
        noise = ndimage.gaussian_filter(noise, sigma=(1.0, 1.0, 0.5))
        noise *= (scale / (np.std(noise) + 1e-8))
        
        image = (image + noise)
        image = ndimage.gaussian_filter(image, sigma=(0.5, 0.5, 0))
        return image

    def crop(self, volume):
        """Removes the safety margin to extract the final shape volume."""
        x0, x1 = self.margin, self.nx - self.margin
        y0, y1 = self.margin, self.ny - self.margin
        z0, z1 = self.margin, self.nz - self.margin
        return volume[x0:x1, y0:y1, z0:z1]

    def getMetrics(self):
        return {
            "shape": self.shape,
            "margin": self.margin,
            "layerRange": self.layerRange,
            "layerThickness": self.layerThickness,
            "foldCount": self.foldCount,
            "foldSigma": self.foldSigma,
            "foldAmplitude": self.foldAmplitude,
            "foldDamping": self.foldDamping,
            "foldBaseShift": self.foldBaseShift,
            "shearOffset": self.shearOffset,
            "shearGradient": self.shearGradient,
            "faultCount": self.faultCount,
            "faultThrow": self.faultThrow,
            "faultDipAngle": self.faultDipAngle,
            "faultRoughness": self.faultRoughness,
            "faultRoughSigma": self.faultRoughSigma,
            "faultDecaySigma": self.faultDecaySigma,
            "faultZoneWidth": self.faultZoneWidth,
            "faultThreshold": self.faultThreshold,
            "faultCurveProb": self.faultCurveProb,
            "faultCurveMax": self.faultCurveMax,
            "waveletFreq": self.waveletFreq,
            "waveletDuration": self.waveletDuration,
            "waveletDt": self.waveletDt,
            "noiseLevel": self.noiseLevel
        }
    
    def print(self):
        print(json.dumps(self.getMetrics(), indent=4))


gen = SyntheticGenerator()

In [3]:
import os, cv2, json, glob
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
import sys, gc
sys.path.append("..")
from Network.index import ModelNetwork

In [4]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

gc.collect()
print(torch.__version__)              # versão do PyTorch
print(torch.cuda.is_available())      # True se detectou a GPU
print(torch.cuda.get_device_name(0))  # nome da GPU

2.7.1+cu118
True
Quadro P6000


In [5]:
baseDir = 'model/'
if not os.path.exists(baseDir):
    print(f"Error: Model dir not found at {baseDir}")

In [6]:
with open(f'{baseDir}/info.json', 'r', encoding='utf-8') as f:
    modelInfo = json.load(f)

modelOptions = modelInfo.get('model', {})
print("Model Options:")
print(json.dumps(modelOptions, indent=4))

network   = ModelNetwork(**modelOptions)
modelData = torch.load(f'{baseDir}/data.pth')

network.model.load_state_dict(modelData['model'])
network.model.eval()

Model Options:
{
    "network": "segresnet",
    "img_size": [
        128,
        128,
        128
    ],
    "classes": 1,
    "channels": 1,
    "dropout": 0.1,
    "num_filters": 32,
    "lr": 0.0005
}


SegResNet(
  (act_mod): ReLU(inplace=True)
  (convInit): Convolution(
    (conv): Conv3d(1, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
  )
  (down_layers): ModuleList(
    (0): Sequential(
      (0): Identity()
      (1): ResBlock(
        (norm1): GroupNorm(8, 32, eps=1e-05, affine=True)
        (norm2): GroupNorm(8, 32, eps=1e-05, affine=True)
        (act): ReLU(inplace=True)
        (conv1): Convolution(
          (conv): Conv3d(32, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
        )
        (conv2): Convolution(
          (conv): Conv3d(32, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1), bias=False)
        )
      )
    )
    (1): Sequential(
      (0): Convolution(
        (conv): Conv3d(32, 64, kernel_size=(3, 3, 3), stride=(2, 2, 2), padding=(1, 1, 1), bias=False)
      )
      (1): ResBlock(
        (norm1): GroupNorm(8, 64, eps=1e-05, affine=True)
        (norm2): GroupNorm(8, 64, eps=1e-05, a

In [7]:
class PredictDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row  = self.df.iloc[index]
        img  = np.load(row.img_path).astype(np.float32)
        mask = np.load(row.mask_path).astype(np.float32)

        img_tensor  = torch.tensor(img,  dtype=torch.float32).unsqueeze(0)
        mask_tensor = torch.tensor(mask, dtype=torch.long).unsqueeze(0)
        return (img_tensor, mask_tensor)

In [8]:
def computeIoU(network, loader):
    network.model.eval()
    network.iou.reset()
    
    with torch.no_grad():
        for imgs, masks in tqdm(loader, desc="Computing IoU"):
            imgs, masks = imgs.to(network.device), masks.to(network.device)
            logits = network.model(imgs)
            
            if network.multiclass:
                preds  = torch.argmax(logits, dim=1)
                target = masks.squeeze(1) if masks.dim() == 5 else masks
                network.iou.update(preds, target)
            else:
                preds = (torch.sigmoid(logits) > 0.5)
                network.iou.update(preds, masks.int())
                
    iouValue = network.iou.compute().item()
    return iouValue

In [9]:
def plotPrediction(index, dataset, network, alpha=0.5, savePath=None):
    img_tensor, mask_tensor = dataset[index]
    network.model.eval()

    with torch.no_grad():
        img_batch = img_tensor.unsqueeze(0).to(network.device)
        logits    = network.model(img_batch)
        pred = torch.argmax(logits, dim=1).squeeze() if network.multiclass else (torch.sigmoid(logits) > 0.5).squeeze().int()

    img_np  = img_tensor.squeeze().cpu().numpy()
    mask_np = mask_tensor.squeeze().cpu().numpy()
    pred_np = pred.cpu().numpy()

    imgNorm = cv2.normalize(img_np, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    imgRgb  = np.stack((imgNorm, imgNorm, imgNorm), axis=-1)
    overlay = imgRgb.copy()
    is_gt   = (mask_np > 0)
    is_pred = (pred_np > 0)
    inters  = (is_gt & is_pred)   # Acertos / Interseção

    overlay[is_gt]   = [0, 0, 255]  # Azul para o Ground Truth
    overlay[is_pred] = [255, 0, 0]  # Vermelho para a Predição
    overlay[inters]  = [0, 255, 0]  # verde onde Predição e GT convergem

    result = (imgRgb * (1 - alpha) + overlay * alpha).astype(np.uint8)
    
    if savePath:
        showTile(result, mask=None, save=savePath)
    else:
        showTile(result, mask=None)

In [ ]:
import random
import string
import shutil
import re
import os
import glob
import pandas as pd
import json
import torch
from torch.utils.data import DataLoader

def get_random_config():
    return {
        'layerRange': (random.randint(50, 150), random.randint(151, 300)),
        'layerThickness': (random.randint(1, 2), random.randint(3, 5)),
        'foldCount': (random.randint(5, 20), random.randint(21, 40)),
        'foldSigma': (random.randint(5, 25), random.randint(26, 60)),
        'foldAmplitude': (-random.randint(5, 20), random.randint(5, 20)),
        'foldDamping': random.uniform(0.5, 2.5),
        'foldBaseShift': (-random.uniform(0.5, 2.0), random.uniform(0.5, 2.0)),
        'shearOffset': (-random.uniform(1.0, 4.0), random.uniform(1.0, 4.0)),
        'shearGradient': (-random.uniform(0.05, 0.2), random.uniform(0.05, 0.2)),
        'faultCount': (4, 7),
        'faultThrow': (random.randint(0, 15), random.randint(16, 30)),
        'faultDipAngle': (random.randint(10, 45), random.randint(46, 85)),
        'faultRoughness': random.uniform(1.0, 5.0),
        'faultRoughSigma': random.uniform(2.0, 7.0),
        'faultDecaySigma': (random.randint(10, 50), random.randint(51, 100)),
        'faultZoneWidth': random.uniform(0.5, 2.0),
        'faultThreshold': random.uniform(0.5, 0.9),
        'faultCurveProb': random.uniform(0.1, 0.6),
        'faultCurveMax': random.uniform(3.0, 10.0),
        'waveletFreq': (random.randint(40, 90), random.randint(91, 150)),
        'waveletDuration': random.uniform(0.04, 0.12),
        'waveletDt': random.uniform(0.001, 0.01),
        'noiseLevel': (0, 0.1)
    }

def run_automation_pipeline(num_variations=3, batch_size=5):
    base_output = 'synthetic'
    os.makedirs(base_output, exist_ok=True)
    
    # Determine starting index by finding the max variation_N in base_output
    existing_dirs = [d for d in os.listdir(base_output) if os.path.isdir(os.path.join(base_output, d)) and d.startswith('variation_')]
    start_idx = 0
    if existing_dirs:
        indices = [int(d.split('_')[1]) for d in existing_dirs if d.split('_')[1].isdigit()]
        if indices:
            start_idx = max(indices) + 1
            
    results_log = []
    
    for i in range(num_variations):
        # 1. Random Parameter Search
        config = get_random_config()
        
        config_id = f"variation_{start_idx + i}"
        config_dir = os.path.join(base_output, config_id)
        os.makedirs(config_dir, exist_ok=True)
        
        print(f"\n[{i+1}/{num_variations}] Generating Data for Config: {config_id}")
        
        try:
            gen = SyntheticGenerator()
            for k, v in config.items():
                setattr(gen, k, v)
                
            # 2. Data Generation
            gen.dataset(n=batch_size, outputDir=config_dir)
            
            # 3. Model Inference Prep
            imgPaths  = sorted(glob.glob(f'{config_dir}/images/*.npy'))
            maskPaths = sorted(glob.glob(f'{config_dir}/masks/*.npy'))
            
            df = pd.DataFrame({'img_path': imgPaths, 'mask_path': maskPaths})
            synthDataset = PredictDataset(df)
            synthLoader  = DataLoader(
                synthDataset, 
                batch_size=1,
                shuffle=False, 
                num_workers=0,
                pin_memory=False
            )
            
            # 4. Evaluation
            iouValue = computeIoU(network, synthLoader)
            print(f"IoU for config {config_id}: {iouValue:.4f}")
            
            # Logging & Visualizations
            plots_dir = os.path.join(config_dir, 'plots')
            os.makedirs(plots_dir, exist_ok=True)
            for j in range(len(synthDataset)):
                plotPath = os.path.join(plots_dir, f'plot_{j:04d}.png')
                plotPrediction(j, synthDataset, network, savePath=plotPath)
                
            metrics = gen.getMetrics()
            metrics['iou'] = iouValue
            
            with open(os.path.join(config_dir, 'config_log.json'), 'w') as f:
                json.dump(metrics, f, indent=4)
                
            results_log.append({
                'config_id': config_id,
                'iou': iouValue
            })
        except Exception as e:
            print(f"Error processing config {config_id}: {e}")
            results_log.append({'config_id': config_id, 'iou': -1, 'error': str(e)})


run_automation_pipeline(num_variations=5000, batch_size=5)


[1/5000] Generating Data for Config: variation_0


Generating dataset:  40%|████      | 2/5 [00:48<01:13, 24.59s/it]